# 02 多工具选择

给 LLM 多个工具，看它如何自动选择正确的那个。

In [ ]:
import sys, json
sys.path.insert(0, '..')

from src.client import get_client

client = get_client()

## 定义多个工具

In [ ]:
def get_weather(city):
    w = {"北京": {"temp": 25, "desc": "晴朗"}, "上海": {"temp": 28, "desc": "多云"}}
    for k in w:
        if k in city:
            return json.dumps(w[k], ensure_ascii=False)
    return json.dumps({"temp": 20, "desc": "未知"}, ensure_ascii=False)

def search_web(query):
    results = {
        "图灵奖": "图灵奖（Turing Award）是计算机科学领域的最高荣誉，由ACM于1966年设立。",
        "Python": "Python 由 Guido van Rossum 于 1991 年发布，是一门广泛使用的编程语言。",
    }
    for k, v in results.items():
        if k in query or query in k:
            return v
    return f"未找到关于'{query}'的结果。"

def calculate(expression):
    try:
        return str(eval(expression))
    except:
        return "计算错误"

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "查询指定城市的天气",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string", "description": "城市名"}},
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "搜索网页，获取关于某个主题的知识或信息",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "搜索关键词"}},
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "执行数学计算，如四则运算、乘方等",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string", "description": "数学表达式"}},
                "required": ["expression"],
            },
        },
    },
]

tool_map = {"get_weather": get_weather, "search_web": search_web, "calculate": calculate}

## 自动路由：LLM 选择正确的工具

In [ ]:
def run(query):
    messages = [{"role": "user", "content": query}]
    response = client.chat.completions.create(
        model="deepseek-chat", messages=messages, tools=tools,
    )
    msg = response.choices[0].message

    if msg.tool_calls:
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            result = tool_map[name](**args)
            print(f"[{name}] {args} → {result}")
            messages.append(msg)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

        final = client.chat.completions.create(
            model="deepseek-chat", messages=messages,
        )
        print(f"回答: {final.choices[0].message.content}")
    else:
        print(f"[无需工具] 回答: {msg.content}")

queries = [
    "上海今天天气怎么样？",
    "帮我搜索一下什么是图灵奖",
    "计算 123 * 456",
    "Python 是什么时候发布的，帮我查一下",
    "你好，今天心情不错",
]
for q in queries:
    print(f">>> {q}")
    run(q)
    print()